In [0]:
  USE CATALOG tibia_analytics;
   USE SCHEMA bronze;
SET TIME ZONE 'UTC';

In [0]:
COPY INTO raw_worlds
FROM (
  SELECT SHA1(
           CONCAT_WS(
             '|',
             _METADATA.FILE_PATH,
             CAST(
               _METADATA.FILE_MODIFICATION_TIME AS STRING
             )
           )
         )                                AS raw_worlds_id,
         FROM_JSON(
           TO_JSON(worlds),
           'STRUCT<
             players_online: INT,
             record_players: INT,
             record_date: STRING,
             regular_worlds: ARRAY<STRUCT<
               name: STRING,
               status: STRING,
               players_online: INT,
               location: STRING,
               pvp_type: STRING,
               premium_only: BOOLEAN,
               transfer_type: STRING,
               battleye_protected: BOOLEAN,
               battleye_date: STRING,
               game_world_type: STRING,
               tournament_world_type: STRING
             >>,
             tournament_worlds: ARRAY<STRUCT<
               name: STRING,
               status: STRING,
               players_online: INT,
               location: STRING,
               pvp_type: STRING,
               premium_only: BOOLEAN,
               transfer_type: STRING,
               battleye_protected: BOOLEAN,
               battleye_date: STRING,
               game_world_type: STRING,
               tournament_world_type: STRING
             >>
           >'
         )                                AS worlds,
         FROM_JSON(
           TO_JSON(information),
           'STRUCT<
             api: STRUCT<
               version: INT,
               release: STRING,
               commit: STRING
             >,
             timestamp: STRING,
             tibia_urls: ARRAY<STRING>,
             status: STRUCT<
               error: INT,
               http_code: INT,
               message: STRING
             >
           >'
         )                                AS information,
         CAST(
           REGEXP_EXTRACT(
             _METADATA.FILE_PATH,
             '([0-9]{4}-[0-9]{2}-[0-9]{2})',
             1
           ) AS DATE
         )                                AS source_file_date,
         _METADATA.FILE_PATH              AS source_file_path,
         _METADATA.FILE_MODIFICATION_TIME AS source_file_modified_at,
         CURRENT_TIMESTAMP()              AS ingested_at
    FROM '/Volumes/tibia_analytics/bronze/raw/worlds'
)
    FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine'   = 'true')
  COPY_OPTIONS ('mergeSchema' = 'false');